In [1]:
import pandas as pd

# Load combined dataset
df = pd.read_csv("../data/combined.csv")
print(df.shape)
print(df.head())
print(df['label'].value_counts())


(99592, 2)
                                                text  label
0  lilly corona estelabalticpinaabcgallerycom loo...      1
1  truett bundynoopputiagccom earth biography htt...      1
2  carlos e r vyjwdtrpcautelefonicanet begin pgp ...      0
3  cristian garry bluewisasanitaircom dear c202f8...      1
4  hi mrs vorke idaho plot 13 victoria garden cit...      1
label
0    50208
1    49384
Name: count, dtype: int64


In [2]:
from sklearn.model_selection import train_test_split

# Split into training data (80%) & testing data (20%)
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
# Split into training data again into training data (90%) and validation set (10%)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.1, random_state=42, stratify=train_labels
)
# Training data (72%) & validation (8%) and testing (20%)
print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")


Train: 71705 | Val: 7968 | Test: 19919


In [3]:
from transformers import BertTokenizer

# Load tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Turn all texts into strings
train_texts = list(map(str, train_texts))
val_texts   = list(map(str, val_texts))
test_texts  = list(map(str, test_texts))

# Tokenize the training, validation and testing data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=256)
val_encodings   = tokenizer(val_texts, truncation=True, padding=True, max_length=256)
test_encodings  = tokenizer(test_texts, truncation=True, padding=True, max_length=256)



In [4]:
import torch
# Custom Dataset class
class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings # Store the tokenized input data
        self.labels = labels # Store the corresponding labels

    def __len__(self):
        return len(self.labels) # Return the total number of samples in the dataset

    def __getitem__(self, idx): # For a given index, retrieve the input tokens and their label
        
        # Each tensor corresponds to one sample (row) from the tokenized data
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels.iloc[idx])) # Add the label
        return item
# Create 3 datasets
train_dataset = EmailDataset(train_encodings, train_labels)
val_dataset = EmailDataset(val_encodings, val_labels)
test_dataset = EmailDataset(test_encodings, test_labels)


In [5]:
from transformers import BertForSequenceClassification

# Load BERT model for binary classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)


2025-10-13 16:06:25.830869: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-13 16:06:26.172311: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-13 16:06:27.593483: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Some weights of BertForSequenceClassification were not initialized from the model checkpoin

In [6]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)   # Predicted class
    
    # Compute precision, recall, and F1 score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")

    # Compute overall accuracy
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [7]:
from transformers import Trainer, TrainingArguments, BertForSequenceClassification
import numpy as np

# Define a function that returns a fresh BERT model for each trial
def model_init():
    return BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

# Define the base training arguments
training_args = TrainingArguments(
    output_dir="./results_tuned", # Where to save model checkpoints & logs
    eval_strategy="epoch", # Evaluate model checkpoints after each epoch
    save_strategy="epoch", # Save model checkpoints after each epoch
    load_best_model_at_end=True, # Load the best model based on eval metric
    metric_for_best_model="f1", # Use F1-score as the selection metric
    logging_dir="./logs_tuned",
)

# Create the Trainer object
trainer = Trainer(
    model_init=model_init,
    args=training_args, # Training configurations
    train_dataset=train_dataset, # Use the Training Dataset
    eval_dataset=val_dataset, # Use the Validation Dataset
    compute_metrics=compute_metrics,
)

# Define how Optuna should explore hyperparameters
def optuna_hp_space(trial):
    return {
        # Sample a learning rate between 1e-5 and 5e-4
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        
        # Try 3, 4, or 5 epochs per training trial
        "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 5),
        
        # Try different batch sizes for training
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [8, 16, 32]
        ),
    }

# Run automatic hyperparameter tuning using Optuna
best_run = trainer.hyperparameter_search(
    direction="maximize", # Success criteria: we want to maximize F1-score
    backend="optuna", # Use Optuna as the tuning backend
    n_trials=10, # Try to rerun the experiment 10 times 
    hp_space=optuna_hp_space,
)

print("Best hyperparameters found:")
print(best_run.hyperparameters)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-10-13 16:06:29,447] A new study created in memory with name: no-name-06745b0b-e861-48c7-87af-746f37bf73c3
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.703500,0.694168,0.504142,0.000000,0.000000,0.000000
2,0.699000,0.693409,0.504142,0.000000,0.000000,0.000000
3,0.698800,0.690176,0.495858,0.495858,1.000000,0.662975
4,0.154000,0.147499,0.968624,0.959980,0.977474,0.968648
5,0.121800,0.107758,0.979167,0.979964,0.977980,0.978971


/home/nmd/projects/phishing-detector/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nmd/projects/phishing-detector/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
[I 2025-10-13 17:56:05,316] Trial 0 finished with value: 3.9160827895304897 and parameters: {'learning_rate': 6.532848178674995e-05, 'num_train_epochs': 5, 'per_device_train_batch_size': 8}. Best is trial 0 with value: 3.9160827895304897.
Some weights of BertForSequenceClassification were not initialized from t

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.695400,0.693688,0.504142,0.000000,0.000000,0.000000
2,0.693300,0.693133,0.504142,0.000000,0.000000,0.000000
3,0.693800,0.693114,0.504142,0.000000,0.000000,0.000000


/home/nmd/projects/phishing-detector/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nmd/projects/phishing-detector/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nmd/projects/phishing-detector/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.123200,0.071113,0.986948,0.983170,0.990635,0.986889


[I 2025-10-14 02:16:25,258] Trial 8 pruned. 
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.720500,0.717236,0.495858,0.495858,1.000000,0.662975
2,0.703600,0.693120,0.504142,0.000000,0.000000,0.000000
3,0.695200,0.693113,0.504142,0.000000,0.000000,0.000000


/home/nmd/projects/phishing-detector/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nmd/projects/phishing-detector/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
[I 2025-10-14 03:21:33,718] Trial 9 pruned. 


Best hyperparameters found:
{'learning_rate': 1.4244780625009774e-05, 'num_train_epochs': 4, 'per_device_train_batch_size': 8}


In [8]:
best_params = best_run.hyperparameters
print(f"Best trial score (F1): {best_run.objective:.4f}")

# Load in using the best hyperparameter found
training_args = TrainingArguments(
    output_dir="./results_final", # Where to save model checkpoints & logs
    eval_strategy="epoch", # Evaluate model checkpoints after each epoch
    save_strategy="epoch", # Save model checkpoints after each epoch
    load_best_model_at_end=True, # Load the best model based on eval metric
    metric_for_best_model="f1",
    learning_rate=best_params["learning_rate"], # Set the learning rate to the best one it has found
    num_train_epochs=int(best_params["num_train_epochs"]), # Set the number of epochs to the best one it has found
    per_device_train_batch_size=int(best_params["per_device_train_batch_size"]), # Set the batch size to the best one it has found
    logging_dir="./logs_final",
)

# Create the Trainer object
trainer = Trainer(
    model=model_init(),
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

Best trial score (F1): 3.9818


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
# Training
trainer.train()

# Save the newly trained Bert model
model.save_pretrained("../models/saved_model")
tokenizer.save_pretrained("../models/saved_model")


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.044400,0.044948,0.990462,0.984981,0.995950,0.990435
2,0.013800,0.044390,0.993850,0.992677,0.994938,0.993806
3,0.000000,0.038626,0.995607,0.996450,0.994685,0.995567
4,0.002200,0.041979,0.995105,0.994939,0.995191,0.995065


('../models/saved_model/tokenizer_config.json',
 '../models/saved_model/special_tokens_map.json',
 '../models/saved_model/vocab.txt',
 '../models/saved_model/added_tokens.json')

In [10]:
# Evaluate the new model with the test_dataset
results = trainer.evaluate(test_dataset)
print("Test Results:", results)


Test Results: {'eval_loss': 0.036433447152376175, 'eval_accuracy': 0.9954817008885988, 'eval_precision': 0.9964492239017957, 'eval_recall': 0.9944315075427761, 'eval_f1': 0.9954393432654303, 'eval_runtime': 101.8949, 'eval_samples_per_second': 195.486, 'eval_steps_per_second': 24.437, 'epoch': 4.0}
